# Cost-Sensitive SVM
Loads `checkpoints/classification_data.npz` → saves `results/result_cs_svm.npz`

In [1]:
import numpy as np, os, time, copy
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
import importlib.util
spec = importlib.util.spec_from_file_location("cfg", "00_config.py")
cfg = importlib.util.module_from_spec(spec); spec.loader.exec_module(cfg)
for a in dir(cfg):
    if not a.startswith('_'): globals()[a] = getattr(cfg, a)

data = np.load(CKPT['clf_data'])
X = data['X_rich']
y = data['y']; groups = data['groups']
subjects = np.load(CKPT['subjects'], allow_pickle=True).tolist()
print(f"X={X.shape} tgt={sum(y==1)} ntgt={sum(y==0)}")

X=(3600, 192) tgt=600 ntgt=3000


In [2]:
from sklearn.model_selection import GridSearchCV

svm = SVC(kernel='rbf', class_weight=CLASS_WEIGHT, probability=True, random_state=42)
grid = GridSearchCV(svm, {'C': [0.01, 0.1, 1.0, 10.0, 100.0]},
    scoring='balanced_accuracy', cv=StratifiedKFold(3, shuffle=True, random_state=42),
    refit=True, n_jobs=-1)
pipe = Pipeline([('scaler', StandardScaler()), ('clf', grid)])

In [3]:
# LOSO
t0 = time.time()
logo = LeaveOneGroupOut()
yt_all, yp_all, scores = [], [], []

for fold, (tr, te) in enumerate(logo.split(X, y, groups)):
    p = copy.deepcopy(pipe)
    p.fit(X[tr], y[tr])
    yp = p.predict(X[te])
    yt_all.extend(y[te]); yp_all.extend(yp)
    ba = balanced_accuracy_score(y[te], yp)
    scores.append(ba)
    print(f"  {subjects[fold]}: {ba:.3f}")

scores = np.array(scores)
print(f"\nmean={np.mean(scores):.3f}±{np.std(scores):.3f} ({time.time()-t0:.0f}s)")

  sub-010: 0.858
  sub-011: 0.655
  sub-012: 0.732
  sub-013: 0.862
  sub-014: 0.837
  sub-015: 0.887
  sub-016: 0.910
  sub-019: 0.653
  sub-020: 0.795
  sub-021: 0.722

mean=0.791±0.090 (260s)


In [4]:
np.savez(result_path('cs_svm'),
    method='cs_svm', per_subject_scores=scores,
    y_true=np.array(yt_all), y_pred=np.array(yp_all),
    elapsed=time.time()-t0, subjects=subjects)
print(f"saved → {result_path('cs_svm')}")
print(classification_report(np.array(yt_all), np.array(yp_all), target_names=['Non-tgt','Target']))

saved → .\results\result_cs_svm.npz
              precision    recall  f1-score   support

     Non-tgt       0.93      0.92      0.93      3000
      Target       0.63      0.66      0.64       600

    accuracy                           0.88      3600
   macro avg       0.78      0.79      0.79      3600
weighted avg       0.88      0.88      0.88      3600

